In [0]:
storage_account_name = "medallionlakeku2026"
client_id = dbutils.secrets.get(scope="medallion-scope", key="sp-client-id")
tenant_id = dbutils.secrets.get(scope="medallion-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="medallion-scope", key="sp-client-secret")

# Configure Spark to authenticate to Azure Data Lake using the Service Principal
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Test the connection by listing the contents of the bronze container
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
# 1. Create a simulated batch of raw data
raw_data = [
    (1001, "CUST-A", 250.50, "2026-11-17"),
    (1002, "CUST-B", 15.00, "2026-11-17"),
    (1003, "CUST-A", 120.00, "2026-11-18"),
    (1004, "CUST-C", 99.99, "2026-11-18"),
    (1005, "CUST-B", None, "2026-11-19") # Simulating a missing value from the source
]

columns = ["order_id", "customer_id", "transaction_amount", "order_date"]

# 2. Build the Spark DataFrame
df_raw = spark.createDataFrame(raw_data, columns)

# 3. View the generated data
display(df_raw)

In [0]:
# Define the exact path in your Data Lake (using the variable from your first cell)
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/raw_sales"

# Write the DataFrame to the Data Lake in Delta format
df_raw.write.format("delta").mode("overwrite").save(bronze_path)

print(f"Success! Raw data is securely stored in the Bronze layer at: {bronze_path}")

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Read the data back out of the Bronze vault
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/raw_sales"
df_bronze = spark.read.format("delta").load(bronze_path)

# 2. Apply Silver Layer Cleansing Rules
# Rule A: Fix the missing transaction amount by replacing 'null' with 0.0
# Rule B: Add a timestamp so we know exactly when this record was cleaned
df_silver = df_bronze.fillna({"transaction_amount": 0.0}) \
                     .withColumn("cleansed_timestamp", current_timestamp())

# 3. View the cleaned data before saving it
display(df_silver)

In [0]:
from delta.tables import DeltaTable

# Define the path to the silver container
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/cleansed_sales"

if DeltaTable.isDeltaTable(spark, silver_path):
    # Incremental Upsert (MERGE) into existing Silver Delta table
    silver_table = DeltaTable.forPath(spark, silver_path)
    (
        silver_table.alias("target")
        .merge(
            df_silver.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Success! Incremental MERGE completed at: {silver_path}")
else:
    # First-time load if the Silver Delta table doesn't exist yet
    df_silver.write.format("delta").mode("overwrite").save(silver_path)
    print(f"Success! Initial Silver Delta table created at: {silver_path}")

In [0]:
from pyspark.sql.functions import sum, round

# 1. Read the cleansed data from the Silver vault
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/cleansed_sales"
df_silver_read = spark.read.format("delta").load(silver_path)

# 2. Apply Gold Layer Business Logic
# Rule: The business wants a report showing the total revenue generated by each customer.
df_gold = df_silver_read.groupBy("customer_id") \
                        .agg(round(sum("transaction_amount"), 2).alias("total_lifetime_value"))

# 3. View the final business-ready data
display(df_gold)

In [0]:
# Define the path to the gold container
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/customer_lifetime_value"

# Write the aggregated DataFrame into the Gold vault
df_gold.write.format("delta").mode("overwrite").save(gold_path)

print(f"Success! Business-ready data is secured in the Gold layer at: {gold_path}")

In [0]:
from delta.tables import DeltaTable

# Reference the Silver Delta table by path
silver_table = DeltaTable.forPath(spark, silver_path)

# 1. Compact small files and Z-Order by customer_id for faster reads
optimize_metrics = silver_table.optimize().executeZOrderBy("customer_id")
display(optimize_metrics)

# 2. Remove stale files older than 7 days (168 hours) to save storage costs
silver_table.vacuum(168)

print(f"Success! OPTIMIZE, ZORDER, and VACUUM completed for: {silver_path}")

In [0]:
# Create a virtual SQL table named 'vw_bronze_sales' pointing to your Data Lake file
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/raw_sales"
spark.read.format("delta").load(bronze_path).createOrReplaceTempView("vw_bronze_sales")

print("Virtual SQL table created successfully!")

In [0]:
%sql
/* Silver Layer: Cleanse the raw data */
SELECT 
    order_id,
    customer_id,
    COALESCE(transaction_amount, 0.0) AS transaction_amount,
    order_date,
    current_timestamp() AS cleansed_timestamp
FROM vw_bronze_sales;

In [0]:
%sql
/* Gold Layer: Aggregate data for business reporting */
SELECT 
    customer_id,
    ROUND(SUM(COALESCE(transaction_amount, 0.0)), 2) AS total_lifetime_value
FROM vw_bronze_sales
GROUP BY customer_id
ORDER BY total_lifetime_value DESC;

In [0]:
# Read the Gold data using your existing Cell 1 credentials
df_gold = spark.read.format("delta").load("abfss://gold@medallionlakeku2026.dfs.core.windows.net/customer_lifetime_value")

# Save it directly into Unity Catalog as a Managed Table
df_gold.write.mode("overwrite").saveAsTable("dbw_medallion_prod.default.gold_customer_lifetime_value")